# Source-Classifier Probe on the Lawgic Topic Encoder

**Purpose.** Test whether the fine-tuned Legal-BERT encoder has learned to encode *which dataset a clause came from* (its "source"), rather than only the legal content of the clause. This is a diagnostic for the **source-style shortcut** risk raised in the panel critique (B4).

---

## The problem, in one paragraph

The fused corpus was built so that source identity correlates almost perfectly with the labels. Three facts cause this:

1. **Mask correlation.** Each source supervises only the topics it covers: a ToS;DR row supervises ~37 topic cells, a CLAUDETTE row only 1-2. The *pattern* of supervised cells half-announces the source.
2. **Label-prior correlation.** Several topics receive positives from only one source (e.g. *trackers* is ToS;DR-only; *limitation of liability* is CLAUDETTE-dominated; *liability cap* / *price changes* are 100 ToS-only). Knowing the source is a strong prior on the label.
3. **Register correlation.** ToS;DR points are curated human quotes; CLAUDETTE is raw sentences; 100 ToS clauses are pulled by annotation code. Different length, punctuation, formatting, tone.

Because only 109 of 26,479 clauses (0.4%) carry more than one source, source and label are nearly separable. Gradient descent can therefore minimise loss by detecting *surface style of the source* and emitting that source's typical label distribution, instead of reading the legal content. At deployment the tool ingests raw ToS text that carries **no source identity**, so any accuracy that rested on the shortcut evaporates. A random clause-level split cannot detect this, because the validation set shares the same source-to-label correlation.

## What this probe does (and does not) prove

This is the **availability** test, not the **harm** test. It measures whether source identity is linearly decodable from the frozen encoder's representation. If it is, the shortcut feature *exists and is available* to the topic head. It does not prove the head *uses* it for topic prediction; that would require the more expensive source-held-out retrain. This probe is the cheap first move: an afternoon of compute that either retires the concern with evidence or tells you the expensive experiment is warranted.

## Design

We extract the pooled `[CLS]` representation (exactly the vector the topic head consumes) for every single-source clause, then train a plain logistic-regression probe to predict the source label from that vector. We report the probe against three references so the number is interpretable:

- **Majority baseline** - always predict the most frequent source (ToS;DR). Sets the floor.
- **Raw (pre-fine-tuning) Legal-BERT** - same probe on the *off-the-shelf* encoder. Isolates how much source-separability the *fine-tuning* added versus what generic legal language modelling already carried.
- **Length-only baseline** - predict source from clause token-length alone. Controls for the most trivial confound: ToS;DR quotes are simply longer. If length alone recovers most of the signal, the "shortcut" is largely length, which any encoder trivially represents.

## How to read the outcome

| Pattern | Interpretation | Action |
|---|---|---|
| Fine-tuned probe macro-F1 ≈ raw ≈ length baseline | Source-separability is generic surface features (mainly length); fine-tuning did not amplify it | Weak shortcut concern. Cite the number, add the acknowledgment sentence, stop. |
| Fine-tuned probe ≫ raw and ≫ length | Fine-tuning carved source identity into the representation beyond trivial features | Strong evidence the ingredient for the shortcut is present. Run the source-held-out retrain to measure actual harm. |
| All probes near the majority floor | Source is not linearly decodable at all | Shortcut unlikely to matter; strongest possible "all clear". |

Runtime on an Apple-Silicon MacBook is a few minutes (one forward pass over ~26k short clauses on MPS). Embeddings are cached to disk, so re-runs are instant.


In [1]:
# --- Cell 1: Imports and configuration ---
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, accuracy_score,
    classification_report, confusion_matrix,
)
from transformers import AutoModel, AutoTokenizer

SEED = 42                      # matches the encoder's training seed
np.random.seed(SEED)
torch.manual_seed(SEED)

# Split proportions used when the encoder was trained (see training_metadata.json).
TEST_SIZE = 0.10
VAL_SIZE = 0.10

MAX_LENGTH = 256               # matches the encoder's training max_length
BATCH_SIZE = 32
BASE_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"


def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "saved_models").is_dir() and (cand / "generated_files").is_dir():
            return cand
    raise FileNotFoundError("Could not locate the lawgic project root from the notebook location.")


PROJECT_ROOT = find_project_root()
CORPUS_CSV = PROJECT_ROOT / "generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv"
FINETUNED_DIR = PROJECT_ROOT / "saved_models/lawgic_classifier_legal-bert_v3"
TOPICS_44_JSON = FINETUNED_DIR / "lawgic_topics_44.json"
CACHE_DIR = PROJECT_ROOT / "outputs/source_probe_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def detect_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = detect_device()
print("Project root:", PROJECT_ROOT)
print("Corpus:", CORPUS_CSV.name, "| exists:", CORPUS_CSV.exists())
print("Fine-tuned encoder:", FINETUNED_DIR.name, "| exists:", FINETUNED_DIR.exists())
print("Device:", DEVICE)

Project root: /Users/riki/Coding Projects/Thesis/lawgic
Corpus: lawgic_multihead_wide.csv | exists: True
Fine-tuned encoder: lawgic_classifier_legal-bert_v3 | exists: True
Device: mps


## Step 1 - Load the corpus and derive a clean source label

We load the 26,479-row training corpus (`lawgic_multihead_wide.csv`). The `sources` column is a JSON list because a clause can be annotated by more than one dataset. For a *source-classification* target each clause needs exactly one label, so we keep only **single-source** rows and drop the 109 multi-source clauses. This removes ambiguity from the target without materially shrinking the data.


In [2]:
# --- Cell 2: Load corpus, keep single-source rows ---
df = pd.read_csv(CORPUS_CSV)
df["source_list"] = df["sources"].apply(json.loads)
df["n_sources"] = df["source_list"].apply(len)

n_total = len(df)
single = df[df["n_sources"] == 1].copy()
single["source"] = single["source_list"].apply(lambda xs: xs[0])
single = single.reset_index(drop=True)

print(f"Total rows: {n_total:,}")
print(f"Single-source rows kept: {len(single):,}  (dropped {n_total - len(single)} multi-source)")
print("\nSource distribution (target classes):")
print(single["source"].value_counts())
print("\nMajority-class share (accuracy floor):",
      round(single['source'].value_counts(normalize=True).max(), 4))

Total rows: 26,479
Single-source rows kept: 26,370  (dropped 109 multi-source)

Source distribution (target classes):
source
tos_dr       21876
claudette     3092
100_tos       1402
Name: count, dtype: int64

Majority-class share (accuracy floor): 0.8296


## Step 2 - Reproduce the encoder's train/val/test split

We rebuild the same stratified 80/10/10 split the encoder used (seed 42, stratified on each clause's primary active topic). The probe is trained on the encoder's **train + validation** rows and evaluated on the encoder's **test** rows.

Why this matters: the encoder applied gradient updates to the train rows, so their representations may be partly memorised. Evaluating the probe on the held-out **test** rows - which the encoder never gradient-trained on - measures whether source identity generalises in the representation, not whether individual rows were memorised.

A note on fidelity: this reproduces the split independently rather than reading back the exact indices. If a handful of rows land differently, the conclusion is unaffected, because the only requirement is that probe-train and probe-test are disjoint. Any residual overlap with the encoder's train set can only *inflate* recoverability, which makes a low probe score a conservative, stronger "all clear".


In [3]:
# --- Cell 3: Build stratify label and reproduce the split ---
topics_44 = json.load(open(TOPICS_44_JSON))
label2id = {t["topic_id"]: t["classifier_id"] for t in topics_44}


def primary_stratify_label(active_ids_json: str) -> str:
    try:
        ids = json.loads(active_ids_json)
    except (TypeError, ValueError):
        ids = []
    for tid in ids:
        if tid in label2id:
            return tid
    return "no_active_topic"


single["stratify_topic"] = single["active_topic_ids"].apply(primary_stratify_label)


def stratify_or_none(labels: pd.Series):
    counts = labels.value_counts()
    if len(counts) < 2 or counts.min() < 2:
        return None
    return labels


train_val_df, test_df = train_test_split(
    single, test_size=TEST_SIZE, random_state=SEED, shuffle=True,
    stratify=stratify_or_none(single["stratify_topic"]),
)
rel_val = VAL_SIZE / (1.0 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df, test_size=rel_val, random_state=SEED, shuffle=True,
    stratify=stratify_or_none(train_val_df["stratify_topic"]),
)

# Probe uses encoder-train + encoder-val as its training pool; encoder-test as held-out eval.
probe_train = pd.concat([train_df, val_df]).reset_index(drop=True)
probe_test = test_df.reset_index(drop=True)

print(f"probe_train rows: {len(probe_train):,}   probe_test rows: {len(probe_test):,}")
print("\nprobe_test source distribution:")
print(probe_test["source"].value_counts())

probe_train rows: 23,733   probe_test rows: 2,637

probe_test source distribution:
source
tos_dr       2184
claudette     297
100_tos       156
Name: count, dtype: int64


## Step 3 - Load the frozen encoders

We load two encoders:

- **Fine-tuned**: the body of your trained classifier (`saved_models/lawgic_classifier_legal-bert_v3`). The saved `model.safetensors` contains the embeddings, transformer stack, and pooler - the classification heads live in separate files and are irrelevant here, since we only need the representation.
- **Raw**: off-the-shelf `nlpaueb/legal-bert-base-uncased`, downloaded once from Hugging Face.

Both are set to `eval()` and never receive gradients. The tokenizer is shared (identical vocabulary).


In [4]:
# --- Cell 4: Load encoders and tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_DIR)

finetuned_encoder = AutoModel.from_pretrained(FINETUNED_DIR).to(DEVICE).eval()
raw_encoder = AutoModel.from_pretrained(BASE_MODEL_NAME).to(DEVICE).eval()

print("Fine-tuned encoder loaded:", finetuned_encoder.config._name_or_path or FINETUNED_DIR.name)
print("Raw encoder loaded:", BASE_MODEL_NAME)
print("Hidden size:", finetuned_encoder.config.hidden_size)

Fine-tuned encoder loaded: /Users/riki/Coding Projects/Thesis/lawgic/saved_models/lawgic_classifier_legal-bert_v3
Raw encoder loaded: nlpaueb/legal-bert-base-uncased
Hidden size: 768


## Step 4 - Extract pooled representations

For each clause we take the **pooled `[CLS]` vector** (`pooler_output`): the `[CLS]` token's final hidden state passed through the pooler dense+tanh layer. This is exactly the 768-dim vector the topic head consumes, so the probe sees precisely what the classifier had available to exploit.

Encoding runs in batches under `torch.no_grad()`. Results are cached to `.npy` keyed by encoder identity and the exact ordered list of texts, so re-running the notebook skips recomputation. Expect a few minutes on first run (MPS), longer on CPU.


In [5]:
# --- Cell 5: Embedding extraction with disk cache ---
@torch.no_grad()
def encode_texts(encoder, texts, tag):
    key = hashlib.md5((tag + "\n".join(texts)).encode("utf-8")).hexdigest()[:16]
    cache_path = CACHE_DIR / f"emb_{tag}_{key}.npy"
    if cache_path.exists():
        print(f"[cache hit] {cache_path.name}")
        return np.load(cache_path)

    vecs = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        enc = tokenizer(
            batch, padding=True, truncation=True, max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        out = encoder(**enc)
        pooled = out.pooler_output            # [batch, 768], the head's input vector
        vecs.append(pooled.cpu().numpy())
        if start % (BATCH_SIZE * 50) == 0:
            print(f"  {tag}: {start + len(batch)}/{len(texts)}")
    emb = np.vstack(vecs).astype(np.float32)
    np.save(cache_path, emb)
    print(f"[cached] {cache_path.name}  shape={emb.shape}")
    return emb


train_texts = probe_train["text"].astype(str).tolist()
test_texts = probe_test["text"].astype(str).tolist()

X_train_ft = encode_texts(finetuned_encoder, train_texts, "finetuned_train")
X_test_ft = encode_texts(finetuned_encoder, test_texts, "finetuned_test")
X_train_raw = encode_texts(raw_encoder, train_texts, "raw_train")
X_test_raw = encode_texts(raw_encoder, test_texts, "raw_test")

y_train = probe_train["source"].to_numpy()
y_test = probe_test["source"].to_numpy()
print("Embedding shapes:", X_train_ft.shape, X_test_ft.shape)

  finetuned_train: 32/23733
  finetuned_train: 1632/23733
  finetuned_train: 3232/23733
  finetuned_train: 4832/23733
  finetuned_train: 6432/23733
  finetuned_train: 8032/23733
  finetuned_train: 9632/23733
  finetuned_train: 11232/23733
  finetuned_train: 12832/23733
  finetuned_train: 14432/23733
  finetuned_train: 16032/23733
  finetuned_train: 17632/23733
  finetuned_train: 19232/23733
  finetuned_train: 20832/23733
  finetuned_train: 22432/23733
[cached] emb_finetuned_train_622dc37b4469a269.npy  shape=(23733, 768)
  finetuned_test: 32/2637
  finetuned_test: 1632/2637
[cached] emb_finetuned_test_6970c99d1ac34a37.npy  shape=(2637, 768)
  raw_train: 32/23733
  raw_train: 1632/23733
  raw_train: 3232/23733
  raw_train: 4832/23733
  raw_train: 6432/23733
  raw_train: 8032/23733
  raw_train: 9632/23733
  raw_train: 11232/23733
  raw_train: 12832/23733
  raw_train: 14432/23733
  raw_train: 16032/23733
  raw_train: 17632/23733
  raw_train: 19232/23733
  raw_train: 20832/23733
  raw_train

## Step 5 - The linear probe

A logistic-regression probe (standardised inputs, `class_weight="balanced"` so the imbalance does not let it win by always predicting ToS;DR). "Linear" is the point: we ask whether source is *trivially* decodable. A powerful non-linear probe could extract source from almost any representation and would not tell us whether the shortcut is easy enough for the topic head to have stumbled onto.

Primary metric is **macro-F1** across the three sources, which weights the minority sources (CLAUDETTE, 100 ToS) equally with ToS;DR. Accuracy is reported too but is misleading here because the majority class alone scores ~0.83. We also print **balanced accuracy**, the per-source classification report, and the confusion matrix.


In [6]:
# --- Cell 6: Fit and evaluate the probe on both encoders ---
def run_probe(X_tr, y_tr, X_te, y_te, name):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0),
    )
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_te)
    macro_f1 = f1_score(y_te, pred, average="macro")
    bal_acc = balanced_accuracy_score(y_te, pred)
    acc = accuracy_score(y_te, pred)
    print(f"\n===== {name} =====")
    print(f"macro-F1: {macro_f1:.4f}   balanced-acc: {bal_acc:.4f}   accuracy: {acc:.4f}")
    labels_sorted = sorted(np.unique(y_te))
    print(classification_report(y_te, pred, labels=labels_sorted, digits=3, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):", labels_sorted)
    print(confusion_matrix(y_te, pred, labels=labels_sorted))
    return {"name": name, "macro_f1": macro_f1, "balanced_acc": bal_acc, "accuracy": acc}


res_ft = run_probe(X_train_ft, y_train, X_test_ft, y_test, "Fine-tuned encoder")
res_raw = run_probe(X_train_raw, y_train, X_test_raw, y_test, "Raw Legal-BERT")


===== Fine-tuned encoder =====
macro-F1: 0.8719   balanced-acc: 0.9087   accuracy: 0.9488
              precision    recall  f1-score   support

     100_tos      0.708     0.872     0.782       156
   claudette      0.828     0.892     0.859       297
      tos_dr      0.989     0.962     0.975      2184

    accuracy                          0.949      2637
   macro avg      0.842     0.909     0.872      2637
weighted avg      0.954     0.949     0.951      2637

Confusion matrix (rows=true, cols=pred): ['100_tos', 'claudette', 'tos_dr']
[[ 136    8   12]
 [  20  265   12]
 [  36   47 2101]]

===== Raw Legal-BERT =====
macro-F1: 0.5754   balanced-acc: 0.6858   accuracy: 0.7569
              precision    recall  f1-score   support

     100_tos      0.263     0.571     0.360       156
   claudette      0.394     0.710     0.507       297
      tos_dr      0.962     0.777     0.859      2184

    accuracy                          0.757      2637
   macro avg      0.540     0.686     

## Step 6 - Reference baselines

Two floors for context. **Majority** always predicts ToS;DR (macro-F1 will be low precisely because it ignores the minority sources). **Length-only** fits the same logistic regression on a single feature, the clause's token count, isolating how much of the source signal is just "ToS;DR quotes are longer". If the fine-tuned probe barely beats length-only, the encoder is not contributing source information beyond a trivial surface statistic.


In [7]:
# --- Cell 7: Majority and length-only baselines ---
# Majority baseline
majority_source = probe_train["source"].value_counts().idxmax()
maj_pred = np.full(len(y_test), majority_source)
res_majority = {
    "name": "Majority (always ToS;DR)",
    "macro_f1": f1_score(y_test, maj_pred, average="macro"),
    "balanced_acc": balanced_accuracy_score(y_test, maj_pred),
    "accuracy": accuracy_score(y_test, maj_pred),
}

# Length-only baseline: token count per clause
def token_lengths(texts):
    return np.array([
        len(tokenizer(t, truncation=True, max_length=MAX_LENGTH)["input_ids"])
        for t in texts
    ]).reshape(-1, 1)

len_train = token_lengths(train_texts)
len_test = token_lengths(test_texts)
len_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)
len_clf.fit(len_train, y_train)
len_pred = len_clf.predict(len_test)
res_length = {
    "name": "Length-only",
    "macro_f1": f1_score(y_test, len_pred, average="macro"),
    "balanced_acc": balanced_accuracy_score(y_test, len_pred),
    "accuracy": accuracy_score(y_test, len_pred),
}
print("Majority baseline macro-F1:", round(res_majority["macro_f1"], 4))
print("Length-only macro-F1:", round(res_length["macro_f1"], 4))

Majority baseline macro-F1: 0.302
Length-only macro-F1: 0.1723


## Step 7 - Summary and decision

The table collates all four. Read it against the decision rule from the top of the notebook: the gap between the **fine-tuned** probe and the **raw** / **length** references is the quantity of interest, not the absolute macro-F1. A small gap means fine-tuning added little source information (weak shortcut concern); a large gap means the encoder learned source identity beyond trivial surface features and the source-held-out retrain is worth running.


In [8]:
# --- Cell 8: Results table + save ---
summary = pd.DataFrame([res_ft, res_raw, res_length, res_majority])[
    ["name", "macro_f1", "balanced_acc", "accuracy"]
].round(4)
summary = summary.rename(columns={
    "name": "Probe", "macro_f1": "macro-F1",
    "balanced_acc": "balanced-acc", "accuracy": "accuracy",
})
display(summary)

gap_vs_raw = res_ft["macro_f1"] - res_raw["macro_f1"]
gap_vs_len = res_ft["macro_f1"] - res_length["macro_f1"]
print(f"\nFine-tuned minus raw   (fine-tuning's added source signal): {gap_vs_raw:+.4f}")
print(f"Fine-tuned minus length (signal beyond trivial length):     {gap_vs_len:+.4f}")

out_csv = CACHE_DIR / "source_probe_results.csv"
summary.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)

print("\nInterpretation guide:")
print("- Small gaps (~0): source-separability is generic/length-driven; weak shortcut concern.")
print("- Large positive gaps: fine-tuning encoded source identity; run the source-held-out retrain.")

,Probe,macro-F1,balanced-acc,accuracy
0,Fine-tuned encoder,0.8719,0.9087,0.9488
1,Raw Legal-BERT,0.5754,0.6858,0.7569
2,Length-only,0.1723,0.3982,0.1779
3,Majority (always ToS;DR),0.3020,0.3333,0.8282



Fine-tuned minus raw   (fine-tuning's added source signal): +0.2965
Fine-tuned minus length (signal beyond trivial length):     +0.6996

Saved: /Users/riki/Coding Projects/Thesis/lawgic/outputs/source_probe_cache/source_probe_results.csv

Interpretation guide:
- Small gaps (~0): source-separability is generic/length-driven; weak shortcut concern.
- Large positive gaps: fine-tuning encoded source identity; run the source-held-out retrain.
